# Folder 03 / file 04 — rollback (failure path, not success OR)

UCI Adult Census Income ([dataset](https://archive.ics.uci.edu/dataset/2/adult)): binary target `income_gt_50k` where **`>50K` = 1** and **`<=50K` = 0**. Fourteen census features; official split is `adult.data` (train) / `adult.test` (holdout).

Inlined replica of `src/n03_prod_cutover/n04_rollback.py`. Run cells **in order** (local or Jobs). Catalog/schema/model/mode come from task env.

Restore previous Adult @prod / serving pin after a failed deploy or canary.


## 1 — Imports


In [ ]:
STOP = False

def _stop(msg: str = "") -> None:
    global STOP
    STOP = True
    print(msg)
    try:
        dbutils.notebook.exit(msg or "ok")  # noqa: F821
    except Exception:
        pass

from src.n03_prod_cutover.n01_approval import resolve_cutover_dest_version
from src.n00_shared.runtime import Settings, configure_mlflow, get_task_value, load_settings, mlflow_client
from src.n00_shared.serving import serving_config, upsert_endpoint


## 2 — `_tag_map`


In [ ]:
def _tag_map(client, name: str, version: str) -> dict:
    mv = client.get_model_version(name, version)
    tags = mv.tags or {}
    if isinstance(tags, dict):
        return tags
    return {t.key: t.value for t in tags}


## 3 — `settings = load_settings()`


In [ ]:
settings = load_settings()


## 4 — `run()` step 1/3


In [ ]:
if not STOP:
    configure_mlflow(settings)
    client = mlflow_client()
    dest = settings.dest_model_name


## 5 — `run()` step 2/3


In [ ]:
if not STOP:
    try:
        current = resolve_cutover_dest_version(settings)
    except Exception:
        current = get_task_value("deploy", "model_version", "")


## 6 — `run()` step 3/3


In [ ]:
if not STOP:
    tags = _tag_map(client, dest, current) if current else {}
    previous = tags.get("previous_prod_version") or get_task_value("deploy", "previous_prod_version", "")
    if not previous:
        print("rollback: no previous @prod; aliases unchanged; revert serving if needed")
        _stop()
    client.set_registered_model_alias(dest, "champion", previous)
    client.set_registered_model_alias(dest, "prod", previous)
    upsert_endpoint(
        settings.endpoint_name,
        serving_config(dest, previous, previous_version=None, canary_percent=100),
    )
    print(f"rollback restored @champion/@prod and serving to v{previous}")
